# Feature Engineering:

Features created:
* temporal features
* transaction amount features
* geographical distance features
* missingness indicators
* card/customer history features
* interaction features
* cautious frequency features

Note: to avoid temporal leakage, history features are calculated only on previous transactions, and frequency features are fitted on the training rows only (time-based split) and then applied to the validation/test rows

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import ks_2samp
from IPython.display import display

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 100)

In [2]:
DATA_DIR = Path("../data/raw")
INTERIM_DIR = Path("../data/interim")

SPARKOV_TRAIN_PATH = DATA_DIR / "fraudTrain.csv"
SPARKOV_TEST_PATH = DATA_DIR / "fraudTest.csv" 

IEEE_TRANSACTION_PATH = DATA_DIR / "train_transaction.csv"
IEEE_IDENTITY_PATH = DATA_DIR / "train_identity.csv"

def read_csv_checked(path):
    if not Path(path).exists():
        raise FileNotFoundError(f"File not found: {Path(path).resolve()}")
    print(f"Reading {path} ...")
    return pd.read_csv(path, low_memory=False)

sparkov_train = read_csv_checked(SPARKOV_TRAIN_PATH)
sparkov_test = read_csv_checked(SPARKOV_TEST_PATH)

ieee_transaction = read_csv_checked(IEEE_TRANSACTION_PATH)
ieee_identity = read_csv_checked(IEEE_IDENTITY_PATH)

Reading ../data/raw/fraudTrain.csv ...
Reading ../data/raw/fraudTest.csv ...
Reading ../data/raw/train_transaction.csv ...
Reading ../data/raw/train_identity.csv ...


## 1. Helper Functions

In [3]:
import numpy as np
import pandas as pd


def add_missingness_features(
    df,
    exclude_cols=None,
    min_missing_rate=0.01
):
    """
    Add missingness indicators only for columns with meaningful missingness.
    """
    df = df.copy()
    exclude_cols = set(exclude_cols or [])

    candidate_cols = [
        c for c in df.columns
        if c not in exclude_cols
        and df[c].isna().mean() >= min_missing_rate
    ]

    missing_features = pd.DataFrame(
        {
            f"{c}_missing": df[c].isna().astype("int8")
            for c in candidate_cols
        },
        index=df.index
    )

    return pd.concat([df, missing_features], axis=1)


def add_log_features(df, columns):
    """
    Add log1p features for non-negative numerical columns.
    """
    df = df.copy()

    for col in columns:
        if col in df.columns:
            values = pd.to_numeric(df[col], errors="coerce")
            values = values.clip(lower=0)
            df[f"{col}_log1p"] = np.log1p(values)

    return df


def add_frequency_features(df, columns, suffix="_freq"):
    """
    Frequency encoding.

    For a train/test workflow, calculate the mapping on the training
    data only and apply it to validation/test data.
    """
    df = df.copy()

    for col in columns:
        if col not in df.columns:
            continue

        frequencies = df[col].value_counts(dropna=False)
        df[f"{col}{suffix}"] = (
            df[col].map(frequencies).fillna(0).astype("float32")
        )

    return df

In [4]:
def fit_frequency_maps(train_df, columns):
    frequency_maps = {}

    for col in columns:
        if col in train_df.columns:
            frequency_maps[col] = train_df[col].value_counts(
                dropna=False
            ).to_dict()

    return frequency_maps


def apply_frequency_maps(df, frequency_maps, suffix="_freq"):
    df = df.copy()

    for col, mapping in frequency_maps.items():
        df[f"{col}{suffix}"] = (
            df[col].map(mapping).fillna(0).astype("float32")
        )

    return df

In [5]:
def haversine_km(lat1, lon1, lat2, lon2):
    """
    Great-circle distance in kilometres between paired coordinates.
    """
    lat1, lon1 = np.radians(lat1), np.radians(lon1)
    lat2, lon2 = np.radians(lat2), np.radians(lon2)

    a = (np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2)

    return 6371.0 * 2 * np.arcsin(np.sqrt(a))


def add_card_window_features(
    df,
    card_col,
    time_col,
    amount_col,
    category_col=None,
    location_cols=None,
    windows=("1h", "24h", "7D", "30D")
):
    """
    Extra past-only card history features (same logic as the team's src/bt4012_fraud/features.py)
    that the dataset-specific history functions do not already create.
    Every value for a transaction at time t only uses the same card's earlier transactions.
    Rolling windows are [t - window, t), so the current transaction is never part of its own history.
    Rows without a card get missing values instead of being pooled into one fake card.
    """
    df = df.copy()

    # ieee-cis TransactionDT is in seconds, sparkov time is already a timestamp
    times = df[time_col]
    if pd.api.types.is_numeric_dtype(times):
        times = pd.to_datetime(times, unit="s")
    else:
        times = pd.to_datetime(times)

    # only rows with a card, in time order within each card
    cards = pd.DataFrame(
        {
            "card": df[card_col],
            "time": times,
            "amount": df[amount_col].astype(float),
            "row": np.arange(len(df)),
        },
        index=df.index
    )
    cards["amount_sq"] = cards["amount"] ** 2

    if category_col is not None:
        cards["category"] = df[category_col]

    if location_cols is not None:
        cards["lat"] = df[location_cols[0]].astype(float)
        cards["lon"] = df[location_cols[1]].astype(float)

    cards = cards[cards["card"].notna()].sort_values(["card", "time", "row"], kind="stable")
    group = cards.groupby("card", sort=False)
    new = pd.DataFrame(index=cards.index)

    # ---------------------------------------------------------
    # First transaction of the card
    # ---------------------------------------------------------
    previous_count = group.cumcount()
    new["is_first_txn"] = (previous_count == 0).astype(int)

    # ---------------------------------------------------------
    # Current amount compared with the card's earlier amounts
    # ---------------------------------------------------------
    past_sum = group["amount"].cumsum() - cards["amount"]
    past_sq_sum = group["amount_sq"].cumsum() - cards["amount_sq"]
    count = previous_count.replace(0, np.nan)

    past_mean = past_sum / count
    past_var = (past_sq_sum / count - past_mean ** 2).clip(lower=0)
    past_std = np.sqrt(past_var * count / (count - 1).replace(0, np.nan))

    new["amt_to_prior_mean"] = cards["amount"] / past_mean.replace(0, np.nan)
    new["amt_prior_zscore"] = (cards["amount"] - past_mean) / past_std.replace(0, np.nan)

    # ---------------------------------------------------------
    # Count, total and mean amount in the last 1h / 24h / 7D / 30D
    # ---------------------------------------------------------
    timed_group = cards.set_index("time").groupby("card", sort=False)["amount"]

    for window in windows:
        rolling = timed_group.rolling(window, closed="left")
        counts = rolling.count().to_numpy()
        sums = rolling.sum().to_numpy()

        new[f"txn_count_{window}"] = np.nan_to_num(counts, nan=0.0)
        new[f"amt_sum_{window}"] = np.nan_to_num(sums, nan=0.0)

        with np.errstate(divide="ignore", invalid="ignore"):
            new[f"amt_mean_{window}"] = np.where(counts > 0, sums / counts, np.nan)

    # ---------------------------------------------------------
    # First time this card uses this category (Sparkov) / ProductCD (IEEE-CIS)
    # ---------------------------------------------------------
    if category_col is not None:
        seen_before = cards.groupby(["card", "category"], sort=False, dropna=False).cumcount()
        new["is_new_category"] = (seen_before == 0).astype(int)

    # ---------------------------------------------------------
    # Distance from the previous merchant and the speed it implies (Sparkov)
    # ---------------------------------------------------------
    if location_cols is not None:
        prev_lat = group["lat"].shift(1)
        prev_lon = group["lon"].shift(1)

        new["km_from_prev_txn"] = haversine_km(cards["lat"], cards["lon"], prev_lat, prev_lon)

        hours = (cards["time"] - group["time"].shift(1)).dt.total_seconds() / 3600
        new["implied_speed_kmh"] = new["km_from_prev_txn"] / hours.replace(0, np.nan)

    # add the new columns back in the original row order
    for col in new.columns:
        df[col] = new[col].astype(float).reindex(df.index)

    return df

## 2. Sparkov Feature engineering:

From EDA, the strongest patterns are:
* transaction amount
* hour of day / late-night activity
* merchant category
* card history
* distance btwn cardholder and merchant location

In [6]:
def engineer_sparkov_base_features(df):
    df = df.copy()

    # ---------------------------------------------------------
    # Timestamp features
    # ---------------------------------------------------------
    df["trans_date_trans_time"] = pd.to_datetime(
        df["trans_date_trans_time"],
        errors="coerce"
    )

    timestamp = df["trans_date_trans_time"]

    df["trans_year"] = timestamp.dt.year.astype("int16")
    df["trans_month"] = timestamp.dt.month.astype("int8")
    df["trans_day"] = timestamp.dt.day.astype("int8")
    df["trans_dayofweek"] = timestamp.dt.dayofweek.astype("int8")
    df["trans_dayofyear"] = timestamp.dt.dayofyear.astype("int16")
    df["trans_hour"] = timestamp.dt.hour.astype("int8")
    df["trans_minute"] = timestamp.dt.minute.astype("int8")

    df["is_weekend"] = (
        timestamp.dt.dayofweek >= 5
    ).astype("int8")

    df["is_night"] = (
        (timestamp.dt.hour >= 22) |
        (timestamp.dt.hour <= 3)
    ).astype("int8")

    df["is_business_hour"] = (
        (timestamp.dt.hour >= 8) &
        (timestamp.dt.hour <= 18)
    ).astype("int8")

    # Cyclical time representations
    df["hour_sin"] = np.sin(
        2 * np.pi * df["trans_hour"] / 24
    )
    df["hour_cos"] = np.cos(
        2 * np.pi * df["trans_hour"] / 24
    )

    df["dow_sin"] = np.sin(
        2 * np.pi * df["trans_dayofweek"] / 7
    )
    df["dow_cos"] = np.cos(
        2 * np.pi * df["trans_dayofweek"] / 7
    )

    # ---------------------------------------------------------
    # Amount features
    # ---------------------------------------------------------
    df["amt_log1p"] = np.log1p(df["amt"].clip(lower=0))
    df["amt_is_round"] = (
        np.isclose(df["amt"] % 1, 0)
    ).astype("int8")

    df["amt_cents"] = (
        np.round((df["amt"] % 1) * 100)
        .astype("int16")
    )

    df["amt_bucket"] = pd.cut(
        df["amt"],
        bins=[-np.inf, 10, 25, 50, 100, 250, 500, 1000, np.inf],
        labels=False
    ).astype("Int64")

    # ---------------------------------------------------------
    # Age features
    # ---------------------------------------------------------
    df["dob"] = pd.to_datetime(df["dob"], errors="coerce")

    df["customer_age"] = (
        timestamp.dt.year - df["dob"].dt.year
        - (
            (timestamp.dt.month < df["dob"].dt.month) |
            (
                (timestamp.dt.month == df["dob"].dt.month) &
                (timestamp.dt.day < df["dob"].dt.day)
            )
        ).astype("int8")
    )

    df["customer_age"] = df["customer_age"].clip(0, 120)

    # ---------------------------------------------------------
    # Geographic features
    # ---------------------------------------------------------
    df["lat_diff"] = df["lat"] - df["merch_lat"]
    df["long_diff"] = df["long"] - df["merch_long"]

    df["lat_diff_abs"] = df["lat_diff"].abs()
    df["long_diff_abs"] = df["long_diff"].abs()

    # Haversine distance in kilometres
    lat1 = np.radians(df["lat"])
    lon1 = np.radians(df["long"])
    lat2 = np.radians(df["merch_lat"])
    lon2 = np.radians(df["merch_long"])

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(lat1)
        * np.cos(lat2)
        * np.sin(dlon / 2) ** 2
    )

    df["card_merchant_distance_km"] = (
        6371 * 2 * np.arcsin(np.sqrt(a))
    )

    df["distance_log1p"] = np.log1p(
        df["card_merchant_distance_km"].clip(lower=0)
    )

    # ---------------------------------------------------------
    # Population-related features
    # ---------------------------------------------------------
    df["city_pop_log1p"] = np.log1p(
        df["city_pop"].clip(lower=0)
    )

    df["amt_per_person"] = (
        df["amt"] /
        (df["city_pop"].replace(0, np.nan))
    )

    df["amt_per_person_log1p"] = np.log1p(
        df["amt_per_person"].clip(lower=0)
    )

    # ---------------------------------------------------------
    # Simple interactions
    # ---------------------------------------------------------
    df["amt_x_night"] = df["amt_log1p"] * df["is_night"]
    df["amt_x_weekend"] = df["amt_log1p"] * df["is_weekend"]
    df["distance_x_night"] = (
        df["distance_log1p"] * df["is_night"]
    )

    return df

In [7]:
sparkov_train_fe = engineer_sparkov_base_features(sparkov_train)
sparkov_test_fe = engineer_sparkov_base_features(sparkov_test)

#### Sparkov Frequency Features
* capture how common an entity appears

In [8]:
sparkov_frequency_cols = [
    "cc_num",
    "merchant",
    "category",
    "city",
    "state",
    "zip",
    "job",
]
# street is not included: every Sparkov card has exactly one street (1:1), so street_freq would be an exact copy of cc_num_freq

In [9]:
# fit the frequency maps on the training block only (earliest 80% of fraudTrain by time) so the validation period and fraudTest do not leak into the counts
sparkov_train_sorted = sparkov_train_fe.sort_values("trans_date_trans_time", kind="stable")
sparkov_fit_rows = sparkov_train_sorted.iloc[: int(len(sparkov_train_sorted) * 0.8)]

sparkov_freq_maps = fit_frequency_maps(
    sparkov_fit_rows,
    sparkov_frequency_cols
)

sparkov_train_fe = apply_frequency_maps(
    sparkov_train_fe,
    sparkov_freq_maps
)

sparkov_test_fe = apply_frequency_maps(
    sparkov_test_fe,
    sparkov_freq_maps
)

#### Sparkov past-only card history features
* as `cc_num` has a long transaction history
* from the card history patterns in notebook 02, fraud comes much sooner after the card's previous transaction (median 1.4 hours vs 4.6 hours for non-fraud, 44% vs 16% within 1 hour) and in short bursts (every fraud burst is over within 3 days), so we add the time since the previous transaction and the number of transactions / amounts over the last 1h, 24h, 7 days and 30 days (`add_card_window_features`), using only earlier transactions

In [10]:
def add_sparkov_past_history_features(df):
    df = df.copy()

    df = df.sort_values(["cc_num", "trans_date_trans_time"], kind="stable")

    group = df.groupby("cc_num", sort=False)

    # Number of earlier transactions by the card
    df["card_previous_txn_count"] = group.cumcount()

    # Time since previous transaction
    # (use the timestamp: unix_time is shifted by 1 extra day for some rows in Sparkov)
    df["seconds_since_card_txn"] = (group["trans_date_trans_time"].diff().dt.total_seconds())

    # Previous amount
    df["previous_card_amt"] = group["amt"].shift(1)

    df["amount_change_from_previous"] = (
        df["amt"] - df["previous_card_amt"]
    )

    df["amount_ratio_to_previous"] = (
        df["amt"] /
        df["previous_card_amt"].replace(0, np.nan)
    )

    # Expanding historical amount statistics
    historical_amt = group["amt"].shift(1)

    df["card_previous_amt_mean"] = (
        historical_amt
        .groupby(df["cc_num"])
        .expanding()
        .mean()
        .reset_index(level=0, drop=True)
        .reindex(df.index)
    )

    df["card_previous_amt_std"] = (
        historical_amt
        .groupby(df["cc_num"])
        .expanding()
        .std()
        .reset_index(level=0, drop=True)
        .reindex(df.index)
    )

    # Restore original row order
    df = df.sort_index()

    return df

note: ensure that test history is only build from the end of training + earlier test transactions

In [11]:
sparkov_all = pd.concat(
    [
        sparkov_train.assign(_dataset="train"),
        sparkov_test.assign(_dataset="test"),
    ],
    axis=0,
    ignore_index=True
)

sparkov_all = engineer_sparkov_base_features(sparkov_all)
sparkov_all = add_sparkov_past_history_features(sparkov_all)

sparkov_all = add_card_window_features(sparkov_all, card_col="cc_num", time_col="trans_date_trans_time", amount_col="amt", category_col="category", location_cols=("merch_lat", "merch_long"))

sparkov_train_fe = (
    sparkov_all[sparkov_all["_dataset"] == "train"]
    .drop(columns="_dataset")
)

sparkov_test_fe = (
    sparkov_all[sparkov_all["_dataset"] == "test"]
    .drop(columns="_dataset")
)

# split column: earliest 80% of fraudTrain = train, latest 20% = validation, fraudTest = test
# (same rows as the ones the frequency maps were fitted on)
order = np.argsort(sparkov_train_fe["trans_date_trans_time"].to_numpy(), kind="stable")
n_fit = int(len(order) * 0.8)
sparkov_split = np.empty(len(order), dtype=object)
sparkov_split[order[:n_fit]] = "train"
sparkov_split[order[n_fit:]] = "validation"
sparkov_train_fe["sparkov_split"] = sparkov_split
sparkov_test_fe["sparkov_split"] = "test"

# the rebuild above starts again from the raw data, so apply the frequency maps (fitted on the training block) again
sparkov_train_fe = apply_frequency_maps(sparkov_train_fe, sparkov_freq_maps)
sparkov_test_fe = apply_frequency_maps(sparkov_test_fe, sparkov_freq_maps)

## 3. IEEE-CIS feature engineering

focus on:
* TransactionDT time features
* ProductCD, card, address and email relationships
* missingness patterns
* identity/device features
* carefully constructed card-history features
* D-column interpretation

#### Merge transaction and identity tables

In [12]:
def merge_ieee_tables(transaction_df, identity_df):
    transaction_df = transaction_df.copy()
    identity_df = identity_df.copy()

    identity_df = identity_df.drop_duplicates(
        subset=["TransactionID"]
    )

    df = transaction_df.merge(
        identity_df,
        on="TransactionID",
        how="left",
        validate="one_to_one",
        suffixes=("", "_identity")
    )

    df["has_identity"] = (
        df["TransactionID"].isin(
            set(identity_df["TransactionID"])
        )
    ).astype("int8")

    return df

In [13]:
ieee_train = merge_ieee_tables(
    ieee_transaction,
    ieee_identity
)

#### IEEE-CIS base features

In [14]:
def engineer_ieee_base_features(df):
    df = df.copy()

    # ---------------------------------------------------------
    # Relative time features
    # ---------------------------------------------------------
    df["relative_day"] = (
        df["TransactionDT"] // 86400
    ).astype("int16")

    df["seconds_in_day"] = (
        df["TransactionDT"] % 86400
    ).astype("int32")

    df["hour"] = (
        df["seconds_in_day"] // 3600
    ).astype("int8")

    df["minute"] = (
        (df["seconds_in_day"] % 3600) // 60
    ).astype("int8")

    # TransactionDT has no clock time, so "night" = the low-activity relative hours 4-10,
    # which also have the highest fraud rates (see the hour chart in notebook 02)
    df["is_night"] = (
        df["hour"].between(4, 10)
    ).astype("int8")

    # no business-hour or weekend flags: relative hours 4-10 are the real night, so relative 8-18 is not
    # business hours, and we do not know which weekday relative day 0 is (the cyclical features below
    # still capture the daily and weekly cycle)

    # Cyclical time
    df["hour_sin"] = np.sin(
        2 * np.pi * df["hour"] / 24
    )
    df["hour_cos"] = np.cos(
        2 * np.pi * df["hour"] / 24
    )

    df["day_sin"] = np.sin(
        2 * np.pi * df["relative_day"] / 7
    )
    df["day_cos"] = np.cos(
        2 * np.pi * df["relative_day"] / 7
    )

    # ---------------------------------------------------------
    # Amount features
    # ---------------------------------------------------------
    df["TransactionAmt_log1p"] = np.log1p(
        df["TransactionAmt"].clip(lower=0)
    )

    df["TransactionAmt_is_round"] = (
        np.isclose(df["TransactionAmt"] % 1, 0)
    ).astype("int8")

    df["TransactionAmt_cents"] = (
        np.round(
            (df["TransactionAmt"] % 1) * 100
        ).astype("int16")
    )

    df["amount_bucket"] = pd.cut(
        df["TransactionAmt"],
        bins=[
            -np.inf, 10, 25, 50, 100,
            250, 500, 1000, 5000, np.inf
        ],
        labels=False
    ).astype("Int64")

    # ---------------------------------------------------------
    # Missingness summary features
    # ---------------------------------------------------------
    feature_cols = [
        c for c in df.columns
        if c not in ["isFraud", "TransactionID"]
    ]

    df["missing_count"] = (
        df[feature_cols].isna().sum(axis=1)
    ).astype("int16")

    df["missing_fraction"] = (
        df[feature_cols].isna().mean(axis=1)
    ).astype("float32")

    # Missingness by feature families
    # (prefix + number only, so DeviceType/DeviceInfo are not counted as D columns;
    # C columns are never missing, so they are left out)
    for prefix in ["D", "V", "M", "id_"]:
        cols = [
            c for c in df.columns
            if c.startswith(prefix)
            and c[len(prefix):].isdigit()
        ]

        if cols:
            df[f"{prefix}missing_count"] = (
                df[cols].isna().sum(axis=1)
            ).astype("int16")

            df[f"{prefix}missing_fraction"] = (
                df[cols].isna().mean(axis=1)
            ).astype("float32")

    # ---------------------------------------------------------
    # Email features
    # ---------------------------------------------------------
    email_cols = [
        c for c in ["P_emaildomain", "R_emaildomain"]
        if c in df.columns
    ]

    if email_cols:
        for col in email_cols:
            df[f"{col}_is_missing"] = (
                df[col].isna()
            ).astype("int8")

            df[f"{col}_provider"] = (
                df[col]
                .astype("string")
                .str.lower()
                .str.split(".", n=1)
                .str[0]
            )

        if (
            "P_emaildomain" in df.columns
            and "R_emaildomain" in df.columns
        ):
            df["email_domain_match"] = (
                df["P_emaildomain"]
                == df["R_emaildomain"]
            ).astype("int8")

    # ---------------------------------------------------------
    # Card and address relationships
    # ---------------------------------------------------------
    card_cols = [
        c for c in [
            "card1", "card2", "card3",
            "card4", "card5", "card6"
        ]
        if c in df.columns
    ]

    if card_cols:
        df["card_missing_count"] = (
            df[card_cols].isna().sum(axis=1)
        ).astype("int8")

    if "addr1" in df.columns and "addr2" in df.columns:
        df["address_missing_count"] = (
            df[["addr1", "addr2"]]
            .isna()
            .sum(axis=1)
            .astype("int8")
        )

        df["address_pair"] = (
            df["addr1"].astype("string")
            + "_"
            + df["addr2"].astype("string")
        )

    if "card4" in df.columns and "card6" in df.columns:
        df["card_type_pair"] = (
            df["card4"].astype("string")
            + "_"
            + df["card6"].astype("string")
        )

    # ---------------------------------------------------------
    # Identity/device features
    # ---------------------------------------------------------
    if "DeviceInfo" in df.columns:
        device_text = (
            df["DeviceInfo"]
            .astype("string")
            .str.lower()
        )

        df["deviceinfo_length"] = (
            device_text.str.len()
        )

        df["deviceinfo_token_count"] = (
            device_text.str.split().str.len()
        ).astype("Int64")

        df["deviceinfo_has_android"] = (
            device_text.str.contains(
                "android",
                na=False
            )
        ).astype("int8")

        df["deviceinfo_has_iphone"] = (
            device_text.str.contains(
                "iphone|ios",
                na=False
            )
        ).astype("int8")

    if "id_30" in df.columns:
        df["id30_os_family"] = (
            df["id_30"]
            .astype("string")
            .str.lower()
            .str.extract(
                r"(windows|ios|android|mac|linux)",
                expand=False
            )
        )

    if "id_31" in df.columns:
        df["id31_browser_family"] = (
            df["id_31"]
            .astype("string")
            .str.lower()
            .str.extract(
                r"(chrome|safari|firefox|edge|opera|ie)",
                expand=False
            )
        )

    if "id_30" in df.columns and "id_31" in df.columns:
        df["os_browser_pair"] = (
            df["id_30"].astype("string")
            + "_"
            + df["id_31"].astype("string")
        )

    # ---------------------------------------------------------
    # D-column derived features
    # ---------------------------------------------------------
    d_cols = [
        c for c in df.columns
        if c.startswith("D")
        and c[1:].isdigit()
    ]

    if d_cols:
        df["D_nonmissing_count"] = (
            df[d_cols].notna().sum(axis=1)
        ).astype("int16")

        df["D_zero_count"] = (
            (df[d_cols] == 0).sum(axis=1)
        ).astype("int16")

    # Features identified by the EDA as more useful
    for col in ["D1", "D3", "D4", "D10", "D15"]:
        if col in df.columns:
            df[f"{col}_missing"] = (
                df[col].isna()
            ).astype("int8")

    if "D1" in df.columns:
        df["card_first_use_day_proxy"] = (
            df["relative_day"] - df["D1"]
        )

    if "D3" in df.columns:
        df["D3_log1p"] = np.log1p(
            df["D3"].clip(lower=0)
        )

    return df

In [15]:
ieee_train_fe = engineer_ieee_base_features(ieee_train)

#### IEEE-CIS time-based split
- the latest 20% of transactions is the held-out test block, and the latest 20% of the rest is the validation block
- the frequency maps below are fitted on the training rows only, so validation/test transactions do not leak into them

In [16]:
order = np.argsort(ieee_train_fe["TransactionDT"].to_numpy(), kind="stable")
n_keep = int(len(order) * 0.8)       # rows before the held-out test block
n_train = int(n_keep * 0.8)          # training rows

ieee_split = np.empty(len(order), dtype=object)
ieee_split[order[:n_train]] = "train"
ieee_split[order[n_train:n_keep]] = "validation"
ieee_split[order[n_keep:]] = "test"
ieee_train_fe["ieee_split"] = ieee_split

display(ieee_train_fe.groupby("ieee_split").agg(
    rows=("TransactionID", "size"),
    first_day=("relative_day", "min"),
    last_day=("relative_day", "max"),
    fraud_rate=("isFraud", "mean")
))

,rows,first_day,last_day,fraud_rate
ieee_split,,,,
test,118108,141,182,0.034409
train,377945,1,109,0.034119
validation,94487,109,141,0.039201


#### IEEE-CIS frequency features

In [17]:
ieee_frequency_cols = [
    "card1",
    "card2",
    "card3",
    "card4",
    "card5",
    "card6",
    "addr1",
    "addr2",
    "P_emaildomain",
    "R_emaildomain",
    "ProductCD",
    "DeviceInfo",
    "id_30",
    "id_31",
    "address_pair",
    "card_type_pair",
]

In [18]:
# fit on the training rows only, then apply to every row
ieee_freq_maps = fit_frequency_maps(
    ieee_train_fe[ieee_train_fe["ieee_split"] == "train"],
    ieee_frequency_cols
)

ieee_train_fe = apply_frequency_maps(
    ieee_train_fe,
    ieee_freq_maps
)

#### IEEE-CIS card ID
* use the card ID reconstructed in `03_ieee_card_reconstruction.ipynb` (`card1 + addr1 + D1n`, refined with `D3`), saved in `data/interim/ieee_card_id.csv`
* transactions without `D1` (0.2%) have no card ID, so they get no card history

In [19]:
card_ids = pd.read_csv(INTERIM_DIR / "ieee_card_id.csv")

ieee_train_fe = ieee_train_fe.merge(card_ids, on="TransactionID", how="left", validate="one_to_one")

print("transactions with a card ID:", round(ieee_train_fe["card_id"].notna().mean() * 100, 2), "%")

transactions with a card ID: 99.79 %


#### Past-only IEEE-CIS history features
* from notebook 03 (Section 4), `D3` looks like the number of days since the card's previous transaction, and it matches the gap to the previous transaction under the base card key for 94.6% of the transactions that can be checked
* from notebook 03 (Section 8), fraud also comes much sooner after the card's previous transaction (median 2.4 hours vs 49.4 hours for non-fraud, 43% vs 25% within 1 hour), and 87.7% of fraud cards have all their fraud within 7 days

Thus,
* for `card_id` and a few other keys (`card1`, `addr1`, the email domains, `DeviceInfo`), we count the earlier transactions and the time since the previous one
* we compare `D3` with the gap to the card's previous transaction (`D3_gap_difference`, `D3_matches_previous_gap`), which flags transactions that do not fit the card's history. Since `card_id` was refined with `D3`, the match is high by construction for most rows
* the card window features (`add_card_window_features`, last 1h/24h/7 days/30 days) are added at the end, grouped by `card_id`

In [20]:
def add_ieee_past_history_features(df):
    df = df.copy()

    sort_cols = ["TransactionDT", "TransactionID"]
    df = df.sort_values(sort_cols)

    history_keys = [
        "card_id",
        "card1",
        "addr1",
        "P_emaildomain",
        "R_emaildomain",
        "DeviceInfo",
    ]

    history_keys = [
        c for c in history_keys
        if c in df.columns
    ]

    for key in history_keys:
        group = df.groupby(key, sort=False)

        df[f"{key}_previous_count"] = (
            group.cumcount()
        )

        previous_time = group["TransactionDT"].shift(1)

        df[f"{key}_seconds_since_previous"] = (
            df["TransactionDT"] - previous_time
        )

    # Compare D3 with the observed previous transaction gap
    if (
        "card_id" in df.columns
        and "D3" in df.columns
    ):
        previous_time = (
            df.groupby("card_id")["TransactionDT"]
            .shift(1)
        )

        observed_gap_days = (
            df["TransactionDT"] - previous_time
        ) / 86400

        df["D3_gap_difference"] = (
            df["D3"] - observed_gap_days
        )

        df["D3_matches_previous_gap"] = (
            np.isclose(
                df["D3"],
                observed_gap_days,
                equal_nan=False,
                atol=1.0
            )
        ).astype("int8")

    return df.sort_index()

In [21]:
ieee_train_fe = add_ieee_past_history_features(ieee_train_fe)

# card history features
ieee_train_fe = add_card_window_features(ieee_train_fe, card_col="card_id", time_col="TransactionDT", amount_col="TransactionAmt", category_col="ProductCD")

print("shape:", ieee_train_fe.shape)

shape: (590540, 531)


## 4. Model inputs
We split the columns into:
- **not model inputs**: identifiers (transaction IDs, card numbers, names, addresses), raw timestamps and time indexes, the label and the split columns. The reason for each group is in the code comments below. Card IDs are only used to build the card-history features

- **candidate features**: everything else (`sparkov_feature_cols`, `ieee_feature_cols`)

These are candidates, not a final selection => feature selection is done separately for each model, using the validation rows only (never the test rows)

In [22]:
# columns that are not model inputs
sparkov_not_features = [
    # row IDs, raw timestamps and calendar position (every validation / test timestamp is new)
    "Unnamed: 0", "trans_num", "unix_time", "trans_date_trans_time", "trans_year", "trans_dayofyear",
    # customer identifiers: each value belongs to 1-2 cards, and no card with fraud in the training period has fraud again later
    "cc_num", "first", "last", "street", "city", "zip", "job", "dob", "lat", "long",
    # frequency versions of customer columns: about half of their values belong to exactly 1 card, so they work like a card ID
    "cc_num_freq", "city_freq", "zip_freq", "job_freq",
    # raw merchant coordinates (used through the distance features), and merchant (adds nothing beyond category, merchant_freq is kept)
    "merch_lat", "merch_long", "merchant",
    # label and split
    "is_fraud", "sparkov_split",
]

sparkov_feature_cols = [c for c in sparkov_train_fe.columns if c not in sparkov_not_features]
sparkov_categorical_cols = [c for c in sparkov_feature_cols if not pd.api.types.is_numeric_dtype(sparkov_train_fe[c])]

print("Sparkov candidate features:", len(sparkov_feature_cols), "| categorical:", sparkov_categorical_cols)

Sparkov candidate features: 62 | categorical: ['category', 'gender', 'state']


In [23]:
# columns that are not model inputs
ieee_not_features = [
    # row ID and time indexes (every validation / test value is later than the training rows)
    "TransactionID", "TransactionDT", "relative_day",
    # card ID and the card's first-use day: both identify the card (card1 + addr1 + first-use day is how card_id was built)ike
    # based on the dataset rule: after a chargeback on a card, later transactions with the same user account, email address or billing address are also labelled fraud (notebook 03, Section 8), and card_id is partly built from these (addr1, D1)
    # the card's behaviour is still used through the past-only history features (e.g. time since the previous transaction, transactions in the last 1h)
    "card_id", "card_first_use_day_proxy",
    # label and split
    "isFraud", "ieee_split",
]

ieee_feature_cols = [c for c in ieee_train_fe.columns if c not in ieee_not_features]
ieee_categorical_cols = [c for c in ieee_feature_cols if not pd.api.types.is_numeric_dtype(ieee_train_fe[c])]

print("IEEE-CIS candidate features:", len(ieee_feature_cols), "| categorical:", len(ieee_categorical_cols))
print(ieee_categorical_cols)

IEEE-CIS candidate features: 524 | categorical: 38
['ProductCD', 'card4', 'card6', 'P_emaildomain', 'R_emaildomain', 'M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8', 'M9', 'id_12', 'id_15', 'id_16', 'id_23', 'id_27', 'id_28', 'id_29', 'id_30', 'id_31', 'id_33', 'id_34', 'id_35', 'id_36', 'id_37', 'id_38', 'DeviceType', 'DeviceInfo', 'P_emaildomain_provider', 'R_emaildomain_provider', 'address_pair', 'card_type_pair', 'id30_os_family', 'id31_browser_family', 'os_browser_pair']


**Takeaway**
- **Candidates**: Sparkov has 62 candidate features (3 categorical: `category`, `gender`, `state`) and IEEE-CIS has 524 (38 categorical)

- **Feature selection** is left to each model and should only use the validation rows. For logistic regression: use the `*_freq` versions of the number-coded columns (`card1`-`card5`, `addr1`, `addr2`), encode the categorical columns (frequency encoding when there are many values, e.g. `DeviceInfo`), and keep only one column of each highly correlated pair from the EDA (e.g. `D4`-`D12`). CatBoost can use the categorical columns directly

- **Splits**: use `sparkov_split` / `ieee_split` for the train / validation / test rows

## 5. Save features
We save the features to `data/processed/`
- `sparkov_features.pkl`: fraudTrain + fraudTest rows, with `sparkov_split`

- `ieee_features.pkl`: all IEEE-CIS rows, with `ieee_split`

- `feature_lists.json`: the candidate feature lists and categorical columns from Section 4

To load them in another notebook:
```python
features = pd.read_pickle("../data/processed/ieee_features.pkl")
feature_lists = json.load(open("../data/processed/feature_lists.json"))

train = features[features["ieee_split"] == "train"]
X_train = train[feature_lists["ieee_feature_cols"]]
y_train = train["isFraud"]
```

In [24]:
import json

PROCESSED_DIR = Path("../data/processed")

sparkov_features = pd.concat([sparkov_train_fe, sparkov_test_fe], ignore_index=True)
sparkov_features.to_pickle(PROCESSED_DIR / "sparkov_features.pkl")
ieee_train_fe.to_pickle(PROCESSED_DIR / "ieee_features.pkl")

feature_lists = {
    "sparkov_feature_cols": sparkov_feature_cols,
    "sparkov_categorical_cols": sparkov_categorical_cols,
    "ieee_feature_cols": ieee_feature_cols,
    "ieee_categorical_cols": ieee_categorical_cols,
}
with open(PROCESSED_DIR / "feature_lists.json", "w") as f:
    json.dump(feature_lists, f, indent=1)

print("Sparkov:", sparkov_features.shape, sparkov_features["sparkov_split"].value_counts().to_dict())
print("IEEE-CIS:", ieee_train_fe.shape, ieee_train_fe["ieee_split"].value_counts().to_dict())

Sparkov: (1852394, 87) {'train': 1037340, 'test': 555719, 'validation': 259335}
IEEE-CIS: (590540, 531) {'train': 377945, 'test': 118108, 'validation': 94487}
